# LightFM — Track B 선호 레시피 추천 (prefer WARP + catalog s_pref)

### Unit 1 - 환경 및 공통 설정

In [ ]:
import sys
from pathlib import Path

import numpy as np
from lightfm import LightFM

PROJECT_ROOT = Path(__import__("os").environ["PROJECT_ROOT"])
sys.path.insert(0, str(PROJECT_ROOT))

from config import load_experiment_config, require_docker_runtime, seed_all, CATALOG_USER_ID

require_docker_runtime()
cfg = load_experiment_config(PROJECT_ROOT)
seed_all(cfg.seed)

print({
    "project_root": str(cfg.project_root),
    "seed": cfg.seed,
    "epochs": cfg.epochs,
    "positive_mode": cfg.positive_mode,
    "mode": cfg.model_mode,
    "excluded_recipe_columns": cfg.excluded_recipe_columns,
})

### Unit 2~4 - 데이터 로드·전처리·Dataset

In [ ]:
from data_io import load_track_b_tables
from preprocess import build_lightfm_ids, prepare_training_frames, validate_id_integrity

review_raw, recipe_raw, alias_df = load_track_b_tables(cfg.data_dir)
recipe_df, review_df = prepare_training_frames(review_raw, recipe_raw, alias_df)
id_report = validate_id_integrity(recipe_df, review_df)
dataset, item_ids, warm_item_ids, cold_item_ids, user_ids = build_lightfm_ids(
    review_df, recipe_df
)

print({"id_report": id_report, "users": len(user_ids), "items": len(item_ids),
        "warm_items": len(warm_item_ids), "cold_items": len(cold_item_ids)})

### Unit 5~5b - interactions·item features

In [ ]:
from preprocess import build_interactions, build_item_features, build_prefer_labels

y_prefer = build_prefer_labels(review_df)
interactions, review_df, _ = build_interactions(
    review_df, dataset, cfg, recipe_prefer_labels=y_prefer
)
item_features, all_feature_names = build_item_features(
    recipe_df, item_ids, dataset, cfg.excluded_recipe_columns
)

print({
    "positive_mode": cfg.positive_mode,
    "y_prefer_pos": int((y_prefer == 1).sum()),
    "shape": interactions.shape,
    "nnz": interactions.nnz,
    "item_feature_nnz": int(item_features.nnz),
    "unique_features": len(all_feature_names),
})

### Unit 6~7 - full train

In [ ]:
train = interactions
model = LightFM(loss="warp", random_state=cfg.seed)
model.fit(
    train,
    item_features=item_features,
    epochs=cfg.epochs,
    num_threads=cfg.num_threads,
)
print(f"trained {cfg.epochs} epochs on prefer WARP matrix (nnz={train.nnz})")

### Unit 11 - catalog export

`scoring.build_export_dataframe` · Go 판정은 `python evaluation.py` (R0~R3 CV)

In [ ]:
import pandas as pd

from data_io import export_recipe_lightfm, write_json_report
from evaluation import catalog_score_sanity
from preprocess import recipe_n_star5_counts
from scoring import aggregate_review_for_export, build_export_dataframe, catalog_predict

review_agg, score_review_formula = aggregate_review_for_export(
    pd.read_csv(cfg.data_files["review"])
)
n_star5 = recipe_n_star5_counts(review_df)
s_pref = catalog_predict(model, dataset, item_ids, item_features, cfg.num_threads)
export_df = build_export_dataframe(
    recipe_df=recipe_df,
    review_agg=review_agg,
    s_pref=s_pref,
    y_prefer=y_prefer,
    n_star5=n_star5,
    warm_item_ids=warm_item_ids,
)

export_path = cfg.outputs_dir / "recipe_lightfm.csv"
export_recipe_lightfm(export_df, export_path)

sanity = catalog_score_sanity(s_pref)
report = {
    "positive_mode": cfg.positive_mode,
    "score_review_formula": score_review_formula,
    "r0_pass": sanity["r0_pass"],
    "n_prefer": int((y_prefer == 1).sum()),
    "export_csv": str(export_path),
}
write_json_report(report, cfg.outputs_dir / "export_report.json")

assert len(export_df) == len(recipe_df) and sanity["r0_pass"]
print(report)
print(f"saved: {export_path.resolve()} ({len(export_df)} rows)")

### Unit 9 - CV Go

`python evaluation.py` → `outputs/prefer_eval_report.json`

In [ ]:
print("CV Go: python evaluation.py")
print(f"export: {cfg.outputs_dir / 'export_report.json'}")